# Раздел 10 · Продакшен: безопасность, масштабирование, мониторинг

In [1]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


/home/dimon/vscode/examples_qdrant/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    OptimizersConfigDiff, CreateAliasOperation, CreateAlias
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [6]:
CONTAINER_NAME = "qdrant_stepik"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6336 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(
    url="http://localhost:6336",
    api_key="Fucking_password_1234567890",
)
print(client.get_collections())


qdrant_stepik
collections=[CollectionDescription(name='day1_semantic_search'), CollectionDescription(name='exercise1_books')]


/tmp/ipykernel_67748/1237198530.py:21: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [8]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29785.05it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


In [9]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


OK: exercise1_books - 87 точек


## Продакшен: снапшоты, алиасы, мониторинг, безопасность

Снова продолжение `BOOKS_COLLECTION` — снапшоты, алиасы и optimizers_config применяются
к уже существующей коллекции без пересоздания. Единственное исключение — демонстрация
`api_key`: чтобы не заблокировать доступ к основному контейнеру, который нужен во всех
остальных частях, для неё поднимается отдельный одноразовый контейнер на другом порту.

### Пример — снапшоты

In [10]:
# снапшот - полная копия коллекции на диске сервера в конкретный момент времени, для бэкапа/переноса
snapshot = client.create_snapshot(collection_name=BOOKS_COLLECTION)
print("создан снапшот:", snapshot.name)

snapshots = client.list_snapshots(collection_name=BOOKS_COLLECTION)
print("все снапшоты коллекции:", [s.name for s in snapshots])


создан снапшот: exercise1_books-1150770739613575-2026-10-31-16-08-05.snapshot
все снапшоты коллекции: ['exercise1_books-1150770739613575-2026-10-31-16-08-05.snapshot']


## Задание 1

Создайте алиас `books_current`, указывающий на `BOOKS_COLLECTION` (тот самый паттерн миграции
без даунтайма — `CreateAliasOperation`/`CreateAlias`). Затем через сам алиас
(а не через `BOOKS_COLLECTION` напрямую) посчитайте `client.count("books_current")` — должно
быть то же число точек, что и в оригинальной коллекции.

**Используйте:** `client.update_collection_aliases(...)`, `CreateAliasOperation(...)`, `CreateAlias(...)`, `client.count(...)`.

In [11]:
client.update_collection_aliases(
    change_aliases_operations=[
        CreateAliasOperation(
            create_alias=CreateAlias(
                collection_name=BOOKS_COLLECTION,   # exercise1_books — куда указывает
                alias_name="books_current",         # как будет называться
            )
        )
    ]
)
count_via_alias = client.count("books_current").count

In [14]:
# ВАШ КОД ЗДЕСЬ
client.update_collection_aliases(
    change_aliases_operations=[
        CreateAliasOperation(
            create_alias=CreateAlias(
                collection_name=BOOKS_COLLECTION,   # exercise1_books — куда указывает
                alias_name="books_current",         # как будет называться
            )
        )
    ]
)
count_via_alias  = client.count("books_current").count

In [15]:
def check_ex1(count_via_alias):
    expected = client.count(BOOKS_COLLECTION).count
    assert count_via_alias == expected, "count через алиас ({}) не совпадает с оригиналом ({})".format(count_via_alias, expected)
    aliases = client.get_collection_aliases(BOOKS_COLLECTION)
    alias_names = [a.alias_name for a in aliases.aliases]
    assert "books_current" in alias_names, "алиас books_current не найден на коллекции"
    print("OK: alias books_current -> {} точек, aliases={}".format(count_via_alias, alias_names))
    print("ОТВЕТ ДЛЯ STEPIK:", count_via_alias)


check_ex1(count_via_alias)


OK: alias books_current -> 87 точек, aliases=['books_current']
ОТВЕТ ДЛЯ STEPIK: 87


### Пример — optimizers_config и мониторинг через /metrics

`optimizers_config` донастраивается точно так же, как quantization в разделе про квантование —
через `update_collection`, без пересоздания:

In [ ]:
client.update_collection(
    collection_name=BOOKS_COLLECTION,
    optimizers_config=OptimizersConfigDiff(default_segment_number=2),
)
info = client.get_collection(BOOKS_COLLECTION)
print("default_segment_number:", info.config.optimizer_config.default_segment_number)

# /metrics отдаёт обычный Prometheus-формат по HTTP, безо всякого специального клиента
import urllib.request
metrics_text = urllib.request.urlopen("http://localhost:6333/metrics").read().decode()
for line in metrics_text.splitlines():
    if line.startswith("collections_total") or ("exercise1_books" in line and "collection_vectors" in line):
        print(line)


Реальный результат: `default_segment_number: 2`, и в `/metrics` видна строка вида
`collection_vectors{collection="exercise1_books",vector=""} 87` — то же число точек, что
вы уже видели через `client.count`, но теперь в формате, который умеет забирать Prometheus
для дашборда, без единого вызова qdrant-client.

### Пример (без самопроверки) — api_key на отдельном контейнере

Поднимем для этого отдельный контейнер на порту 7333, чтобы не блокировать основной
`qdrant_lecture`, который используется во всех остальных частях ноутбука:

In [ ]:
def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker rm -f qdrant_secure_demo")  # на случай, если остался от прошлого запуска
sh(
    # QDRANT__SERVICE__API_KEY - переменная окружения, которая включает обязательную авторизацию по ключу
    "docker run -d --name qdrant_secure_demo -p 7333:6333 "
    "-e QDRANT__SERVICE__API_KEY=student-secret-key qdrant/qdrant"
)

import time
time.sleep(3)  # даём контейнеру время подняться, прежде чем стучаться в него

import urllib.request


def status_code(url, headers=None):
    req = urllib.request.Request(url, headers=headers or {})
    try:
        return urllib.request.urlopen(req).status
    except urllib.error.HTTPError as e:
        return e.code


print("без ключа:      ", status_code("http://localhost:7333/collections"))
print("с неверным ключом:", status_code("http://localhost:7333/collections", {"api-key": "wrong"}))
print("с верным ключом:  ", status_code("http://localhost:7333/collections", {"api-key": "student-secret-key"}))

sh("docker rm -f qdrant_secure_demo")  # уборка сразу же, это была только демонстрация


Реальный результат: `401`, `401`, `200` — без ключа или с неверным ключом сервер отказывает
ещё до того, как запрос доходит до логики коллекций, с верным ключом всё работает как обычно.

In [16]:
!uv pip install langchain-qdrant

Using Python 3.14.2 environment at: /home/dimon/vscode/examples_qdrant/.venv
Resolved 42 packages in 1.17s                                        
Prepared 11 packages in 722ms                                                zstandard            ------------------------------ 4.77 MiB/5.30 MiB           zstandard            ------------------------------ 688.00 KiB/5.30 MiB         zstandard            ------------------------------ 496.00 KiB/5.30 MiB         zstandard            ------------------------------ 288.00 KiB/5.30 MiB         zstandard            ------------------------------ 272.00 KiB/5.30 MiB         zstandard            ------------------------------ 128.00 KiB/5.30 MiB         zstandard            ------------------------------ 16.00 KiB/5.30 MiB          zstandard            ------------------------------     0 B/5.30 MiB            
Installed 17 packages in 9ms3                               
 + distro==1.9.0
 + httpcore2==2.16.0
 + httpx2==2.16.0
 + jsonpatch==1.38

In [17]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

ModuleNotFoundError: No module named 'langchain_openai'